In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
from openai import OpenAI

In [ ]:
#In this cell, I am going to set up ollama model
ollama_url= 'http://localhost:11434/v1'
ollama=OpenAI(api_key='ollama', base_url=ollama_url)
model='llama3.2-vision'

In [ ]:
img_path = Path("images") / "food_1.jpg"


img = Image.open(img_path)

# Inspect bytes + metadata
img_bytes = img_path.read_bytes()
print(f"Image path: {img_path}")
print(f"Image size: {len(img_bytes)} bytes")
print(f"Image dimensions: {img.size}")
print(f"Image format: {img.format}")

# Display in notebook
display(img)
#store the image in variable
image = img


In [ ]:
#We are going to encode the image that is going to be either a file path or a PIL image to a base64 encoded string.
import os
import io
import base64

def encode_image(image_input):
    # Read the image from the specified path or use a PIL Image object
    if isinstance(image_input, (str, Path)):
        image_path = str(image_input)
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        with open(image_path, 'rb') as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')

    elif isinstance(image_input, Image.Image):
        buffer = io.BytesIO()
        image_format = image_input.format or "JPEG"
        image_input.save(buffer, format=image_format)
        return base64.b64encode(buffer.getvalue()).decode('utf-8')

    else:
        raise TypeError("Invalid input type. Expected a string, Path, or PIL Image object.")


In [ ]:
def llama_vision(client, image, prompt, model = model, max_tokens=2048):
    """Send an image to the Llama Vision API and return the response."""
    # Encode the image as base64

    image_base64= encode_image(image)

    # Create the request payload
    try:
        payload = {
        "model": model,
        "messages": [
        {"role": "system", "content": "You are an image-capable nutrition analyst."},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
            ]
        }
    ],
    "max_tokens": max_tokens
    }       
        # Send the request to the Llama Vision API
        response = client.chat.completions.create(**payload)
        return response.choices[0].message.content
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


In [ ]:
prompt =""" 
context= You are a helpful assistant that can analyze images and give the analysis of the image.
instruction = Analyze the image and give the amount of calories it contains as well as the nutritional value.
Answer only Json and give the answer in the following format:

input= The image the user shares.
output= 
Provide the following estimated nutritional information for a typical serving size or per 100g:
- food_name (string)
- serving_description (string, e.g., '1 slice', '100g', '1 cup')
- calories (float)
- fat_grams (float)
- protein_grams (float)
- sugar_grams (float)
- fiber_grams (float)
- confidence_level (string: 'High', 'Medium', or 'Low')
**IMPORTANT**: Only respond with the JSON object containing the nutritional information.

Example valid JSON response:
{
  "food_name": "Banana",
  "serving_description": "1 medium banana (approx 118g)",
  "calories": 105.0,
  "fat_grams": 0.4,
  "protein_grams": 1.3,
  "confidence_level": "High"
}
"""
print(f"Prompt: {prompt}")

In [ ]:
print("Sending image to Llama Vision API...")
response = llama_vision(ollama, image, prompt)
print("Response from Llama Vision API:")
print(response) 

In [ ]:
# Install Gradio if needed
%pip install --quiet gradio

import gradio as gr


def llama_vision_gradio(image, prompt_text):
    if image is None:
        return "No image provided. Please upload an image."
    return llama_vision(ollama, image, prompt_text)


def launch_gradio_app():
    with gr.Blocks() as demo:
        gr.Markdown("# Calorie Tracker (Llama Vision)")
        gr.Markdown("Upload a food image, then ask the model to analyze calories and nutrition.")

        with gr.Row():
            image_input = gr.Image(type="pil", label="Upload food image")
            prompt_input = gr.Textbox(
                lines=6,
                value=prompt,
                label="Prompt",
                placeholder="Ask the model to analyze calories and nutrition from the image."
            )

        output_text = gr.Textbox(
            label="Model Output",
            lines=3,
            max_lines=30,
            interactive=False,
        )
        run_button = gr.Button("Analyze Image")

        run_button.click(
            llama_vision_gradio,
            inputs=[image_input, prompt_input],
            outputs=output_text,
        )

    demo.launch(share=False)


launch_gradio_app()